In [1]:
# importar la api key y el tenant
import os
import pandas as pd
import json
import requests

CONFIG_PATH = os.path.join("..","..","config.json")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

c:\Users\dblan\anaconda3\envs\Assistants_con\lib\site-packages\requests\__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [9]:
TENANT = config["Qlik_connection"]["Qlik_tenant"]
API_KEY= config["Qlik_connection"]["Qlik_4s"]

base = f"https://{TENANT}/api/v1"
headers = {"Authorization": f"Bearer {API_KEY}"}

In [10]:
resp = requests.get(f"{base}/apps", headers=headers)
print("Status code:", resp.status_code)
print("Response body:", resp.text)

Status code: 200
Response body: {"data":[{"attributes":{"id":"0d83ade6-9d52-4da1-8448-59341f421d08","name":"REDI LATAM","description":"","thumbnail":"/api/v1/apps/0d83ade6-9d52-4da1-8448-59341f421d08/media/files/1Monterrey.png","lastReloadTime":"2025-07-19T10:56:16.258Z","createdDate":"2025-02-06T19:05:30.048Z","modifiedDate":"2025-07-19T10:56:39.854Z","owner":"auth0|47f8a54afdebc603e7753184154e0aeaedfc1d3a970962d1355e2398671868ad","ownerId":"677dc3b540acfc79eec6dac3","dynamicColor":"","published":false,"publishTime":"","custom":{},"hasSectionAccess":true,"encrypted":true,"originAppId":"","isDirectQueryMode":false,"usage":"ANALYTICS","spaceId":"67a4fa985f80476eb2be74b5","_resourcetype":"app"},"privileges":[],"create":[]},{"attributes":{"id":"0e57ff25-9f70-4973-8134-78add8ada5a6","name":"4S Interno - Definición","description":"","thumbnail":"","lastReloadTime":"2025-07-19T07:11:59.817Z","createdDate":"2025-02-23T03:15:57.437Z","modifiedDate":"2025-07-19T07:12:25.775Z","owner":"auth0|47f

In [ ]:
apps = resp.json().get("data", [])
print("Apps encontradas:", [a["attributes"]["name"] for a in apps])
print("IDs:", [a["attributes"]["id"] for a in apps])

Apps encontradas: ['REDI LATAM', '4S Interno - Definición', 'App Analyzer', '4S Interno - Auditoría', 'Reload Analyzer', '4S Interno - Auditoria Big Data', 'DEV - Investigación', '4S Corporativo', 'REDI - Testing Stage', 'Access Evaluator', '4S Interno - Gran Reporte de Verticalización', '4S Demo ELDI', 'REDI Database Extraction', 'REDI Mx - backup', 'REDI - Retail', 'Report Analyzer', 'TECH - Widgets', 'Answers Analyzer', 'Consumption report 2025-03-21', 'TECH - Camilo B.', 'test_core_data', 'REDI - Real Estate Data Insights', 'TECH Real Estate Data Insights - OV', 'REDI - Pruebas Tech', 'Script - 4S Interno - Inf Secundaria', '4S Interno - Opinion de Valor', 'Info Secundaria Loader', 'REDI Financiero', 'AI_chatbot_test', '4S Interno - Test Relacion Geografica inf secundaria', '4S Interno - Inf Sec (no usar)', 'Automation Analyzer', 'Test', 'Access Evaluator_Tenant ant.', '4S Interno - Inf Secundaria', 'TECH - Estudio Vertical (Pruebas)', 'TECH - REDI Interno', 'Consumption report 202

## El script que sigue, se conecta a qlik, y extrae todas las filas de una tabla dado el id especifico de esa tabla.
### Comentado tiene la opcion de crear un archivo en .csv para verificar la tabla con los titulos de las columnas.
### Finalmente imprime las primeras 5 filas de la tabla extraida

In [29]:
import websocket
import json
import pandas as pd

# Global variables
table_handle = None
all_rows = []
page_top = 0
page_size = 1000
total_rows = None
column_names = None  # Will be set dynamically

APP_ID = config["Qlik_connection"]["AI_chatbot_test"]
url = f"wss://{TENANT}/app/{APP_ID}"

def on_message(ws, msg):
    global table_handle, all_rows, page_top, total_rows, column_names

    resp = json.loads(msg)
    msg_id = resp.get("id")
    print(f"DEBUG – ID recibido: {msg_id}")

    # 1) OpenDoc → GetObject (table)
    if msg_id == 1:
        doc_handle = resp["result"]["qReturn"]["qHandle"]
        ws.send(json.dumps({
            "jsonrpc": "2.0", "id": 2, "handle": doc_handle,
            "method": "GetObject", "params": {"qId": "BCSxDL"}
        }))

    # 2) GetObject → save table_handle → GetLayout
    elif msg_id == 2:
        table_handle = resp["result"]["qReturn"]["qHandle"]
        ws.send(json.dumps({
            "jsonrpc": "2.0", "id": 3, "handle": table_handle,
            "method": "GetLayout", "params": {}
        }))

    # 3) GetLayout → extract size & column names → first GetHyperCubeData
    elif msg_id == 3:
        layout = resp["result"]["qLayout"]["qHyperCube"]
        size = layout["qSize"]
        total_rows = size["qcy"]

        # Dynamic column names from dimensionInfo & measureInfo
        dim_info  = layout["qDimensionInfo"]
        meas_info = layout["qMeasureInfo"]
        dim_names  = [d["qFallbackTitle"] for d in dim_info]
        meas_names = [m["qFallbackTitle"] for m in meas_info]
        column_names = dim_names + meas_names

        pages = [{
            "qTop":  0,
            "qLeft": 0,
            "qHeight": min(page_size, total_rows),
            "qWidth": size["qcx"]
        }]
        payload = {
            "jsonrpc": "2.0",
            "id":       4,
            "handle":   table_handle,
            "method":   "GetHyperCubeData",
            "params":  ["/qHyperCubeDef", pages]
        }
        print(f"DEBUG – Enviando GetHyperCubeData: top=0, height={pages[0]['qHeight']}")
        ws.send(json.dumps(payload))

    # 4) GetHyperCubeData → accumulate pages & paginate or finish
    elif msg_id == 4:
        page = resp["result"]["qDataPages"][0]
        matrix = page["qMatrix"]
        all_rows.extend([[c.get("qText","") for c in row] for row in matrix])
        print(f"DEBUG – Recibida página top={page_top}, filas={len(matrix)}")

        if page_top + page_size < total_rows:
            page_top += page_size
            next_pages = [{
                "qTop":    page_top,
                "qLeft":   0,
                "qHeight": min(page_size, total_rows - page_top),
                "qWidth":  page["qArea"]["qWidth"]
            }]
            print(f"DEBUG – Enviando GetHyperCubeData: top={page_top}, height={next_pages[0]['qHeight']}")
            ws.send(json.dumps({
                "jsonrpc": "2.0",
                "id":       4,
                "handle":   table_handle,
                "method":   "GetHyperCubeData",
                "params":  ["/qHyperCubeDef", next_pages]
            }))
        else:
            # All pages received → build DataFrame & save
            df = pd.DataFrame(all_rows, columns=column_names)
            print("Vista previa del DataFrame:")
            print(df.head())
            #df.to_csv("tabla_qlik.csv", index=False)
            #print(">> Archivo guardado: tabla_qlik.csv")
            ws.close()

def on_open(ws):
    ws.send(json.dumps({
        "jsonrpc":"2.0","id":1,"handle":-1,
        "method":"OpenDoc","params":{"qDocName":APP_ID}
    }))

ws = websocket.WebSocketApp(
    url,
    on_message=on_message,
    on_open=on_open,
    header=[f"Authorization: Bearer {API_KEY}", "Sec-WebSocket-Protocol: qlik.api"]
)

ws.run_forever(sslopt={"cert_reqs": 0})


DEBUG – ID recibido: None
DEBUG – ID recibido: 1
DEBUG – ID recibido: 2
DEBUG – ID recibido: 3
DEBUG – Enviando GetHyperCubeData: top=0, height=1000
DEBUG – ID recibido: 4
DEBUG – Recibida página top=0, filas=1000
DEBUG – Enviando GetHyperCubeData: top=1000, height=1000
DEBUG – ID recibido: 4
DEBUG – Recibida página top=1000, filas=1000
DEBUG – Enviando GetHyperCubeData: top=2000, height=1000
DEBUG – ID recibido: 4
DEBUG – Recibida página top=2000, filas=1000
DEBUG – Enviando GetHyperCubeData: top=3000, height=1000
DEBUG – ID recibido: 4
DEBUG – Recibida página top=3000, filas=1000
DEBUG – Enviando GetHyperCubeData: top=4000, height=1000
DEBUG – ID recibido: 4
DEBUG – Recibida página top=4000, filas=1000
DEBUG – Enviando GetHyperCubeData: top=5000, height=1000
DEBUG – ID recibido: 4
DEBUG – Recibida página top=5000, filas=1000
DEBUG – Enviando GetHyperCubeData: top=6000, height=1000
DEBUG – ID recibido: 4
DEBUG – Recibida página top=6000, filas=1000
DEBUG – Enviando GetHyperCubeData: t

False